# 04 EDA

Before clustering anything, look at what we have. The blobs in week 0 were tidy: two
features, both with a similar scale, clusters you could see.

Real data is not like that. This notebook is about understanding the data a bit more **before** we run the clustering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

CLASS_NAMES = {1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine",
               4: "Cottonwood/Willow", 5: "Aspen", 6: "Douglas-fir",
               7: "Krummholz"}

d = np.load("data/covertype_sample.npz")
X, y = d["X"], d["y"]
FEATURES = list(d["feature_names"])

print(f"X {X.shape}   y {y.shape}   {len(FEATURES)} features")

### 54 features

Here is the range of each feature

In [ ]:
summary = pd.DataFrame({
    "min": X.min(axis=0),
    "max": X.max(axis=0),
    "mean": X.mean(axis=0),
    "std": X.std(axis=0),
}, index=FEATURES)

summary.round(2).loc[
    ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
     "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
     "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
     "Horizontal_Distance_To_Fire_Points"]
]

### The binary features

The 44 wilderness/soil flags are one-hot columns: each plot belongs to exactly one
wilderness area and one soil type. Most soil types are rare, which means most of those
columns are almost entirely zeros.

In [ ]:
binary = [f for f in FEATURES if f.startswith(("Wilderness_Area_", "Soil_Type_"))]
idx = [FEATURES.index(f) for f in binary]
prevalence = pd.Series(X[:, idx].mean(axis=0), index=binary).sort_values(ascending=False)

print(f"{len(binary)} binary columns")
print(f"  always zero      : {(prevalence == 0).sum()}")
print(f"  present in <1%   : {((prevalence > 0) & (prevalence < 0.01)).sum()}")
print(f"  present in >50%  : {(prevalence > 0.5).sum()}")
print()
prevalence.head(10).round(4)

### A few interesting features

Elevation separates the species best of anything here. Aspect and slope much less so.

In [ ]:
show = ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Roadways"]

fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, name in zip(axes.ravel(), show):
    ax.hist(X[:, FEATURES.index(name)], bins=50, color="0.6", edgecolor="white")
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Same features split by species

Histograms of the pooled data hide the classes. Split them and you can see that the species
really do live at different elevations but with a lot of overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
elev = X[:, FEATURES.index("Elevation")]

for c in sorted(np.unique(y)):
    ax.hist(elev[y == c], bins=60, histtype="step", linewidth=1.5,
            label=CLASS_NAMES[int(c)])
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("plots")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Are the features correlated?

In [ ]:
numeric = FEATURES[:10]
C = np.corrcoef(X[:, :10], rowvar=False)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(C, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels(numeric, rotation=90, fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(numeric, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Correlation, 10 numeric features")
plt.tight_layout()
plt.show()